# 地勘阶段 BIM

本文件包含全部辅助函数。输入与输出均直接放在本 Notebook 所在目录，不创建子目录，也不依赖外部 Python 脚本。

从同级 tunnel_axis.json、tunnel_profile.json 和 Datasource_dikan_new.xlsx 读取数据，生成可视化轴线和 IFC4x3 参数化路线两种地勘 BIM，并读取路线信息。

依赖：IfcOpenShell、numpy、pandas、openpyxl。


In [1]:
from pathlib import Path

PROJECT_DIR = Path.cwd().resolve()
if not (PROJECT_DIR / 'dikan2bim_new.ipynb').is_file():
    raise RuntimeError("请从此 Notebook 所在目录启动内核。")
DATA_DIR = PROJECT_DIR
OUTPUT_DIR = PROJECT_DIR


## 内置 IFC 辅助函数


In [2]:
# 地勘信息生成bim模型所需的函数
import ifcopenshell
from ifcopenshell import api
import math

O = 0., 0., 0.
X = 1., 0., 0.
Y = 0., 1., 0.
Z = 0., 0., 1.


# IFC template creation
def create_template(filename, timestamp, timestring, creator, organization, application, application_version,
                    project_globalid, project_name):
    filename = str(filename).replace('\\', '/').rsplit('/', 1)[-1]
    template = """ISO-10303-21;
    HEADER;
    FILE_DESCRIPTION(('ViewDefinition [CoordinationView]'),'2;1');
    FILE_NAME('%(filename)s','%(timestring)s',('%(creator)s'),('%(organization)s'),'%(application)s','%(application)s','');
    FILE_SCHEMA(('IFC4X3'));
    ENDSEC;
    DATA;
    #1=IFCPERSON($,$,'%(creator)s',$,$,$,$,$);
    #2=IFCORGANIZATION($,'%(organization)s',$,$,$);
    #3=IFCPERSONANDORGANIZATION(#1,#2,$);
    #4=IFCAPPLICATION(#2,'%(application_version)s','%(application)s','');
    #5=IFCOWNERHISTORY(#3,#4,$,.ADDED.,$,#3,#4,%(timestamp)s);
    #6=IFCDIRECTION((1.,0.,0.));
    #7=IFCDIRECTION((0.,0.,1.));
    #8=IFCCARTESIANPOINT((0.,0.,0.));
    #9=IFCAXIS2PLACEMENT3D(#8,#7,#6);
    #10=IFCDIRECTION((0.,1.,0.));
    #11=IFCGEOMETRICREPRESENTATIONCONTEXT($,'Model',3,1.E-05,#9,#10);
    #12=IFCDIMENSIONALEXPONENTS(0,0,0,0,0,0,0);
    #13=IFCSIUNIT(*,.LENGTHUNIT.,$,.METRE.);
    #14=IFCSIUNIT(*,.AREAUNIT.,$,.SQUARE_METRE.);
    #15=IFCSIUNIT(*,.VOLUMEUNIT.,$,.CUBIC_METRE.);
    #16=IFCSIUNIT(*,.PLANEANGLEUNIT.,$,.RADIAN.);
    #17=IFCMEASUREWITHUNIT(IFCPLANEANGLEMEASURE(0.017453292519943295),#16);
    #18=IFCCONVERSIONBASEDUNIT(#12,.PLANEANGLEUNIT.,'DEGREE',#17);
    #19=IFCUNITASSIGNMENT((#13,#14,#15,#18));
    #20=IFCPROJECT('%(project_globalid)s',#5,'%(project_name)s',$,$,$,$,(#11),#19);
    #21=IFCAXIS2PLACEMENT3D(#8,#7,#6);
    #22=IFCGEOMETRICREPRESENTATIONCONTEXT('Model','Model',3,1.E-05,#21);
    #23=IFCSIUNIT(.PLANEANGLEUNIT.,.DEGREE.,$);
    #24=IFCUNITASSIGNMENT((#13,#14,#15,#23));
    ENDSEC;
    END-ISO-10303-21;
    """ % locals()  
    return template


def create_ifcaxis2placement(ifcfile, point=O, dir1=Z, dir2=X):
    point = ifcfile.createIfcCartesianPoint(point)
    dir1 = ifcfile.createIfcDirection(dir1)
    dir2 = ifcfile.createIfcDirection(dir2)
    axis2placement = ifcfile.createIfcAxis2Placement3D(point, dir1, dir2)
    return axis2placement


def create_ifcaxis2placement_2d(ifcfile, point=O, dir1=Z):
    point = ifcfile.createIfcCartesianPoint(point)
    dir1 = ifcfile.createIfcDirection(dir1)
    axis2placement = ifcfile.createIfcAxis2Placement2D(point, dir1)
    return axis2placement


def create_ifclocalplacement(ifcfile, point=O, dir1=Z, dir2=X, relative_to=None):
    axis2placement = create_ifcaxis2placement(ifcfile, point, dir1, dir2)
    ifclocalplacement2 = ifcfile.createIfcLocalPlacement(relative_to, axis2placement)
    return ifclocalplacement2


def create_ifcpolyline(ifcfile, point_list):
    ifcpts = []
    for point in point_list:
        point = ifcfile.createIfcCartesianPoint(point)
        ifcpts.append(point)
    polyline = ifcfile.createIfcPolyLine(ifcpts)
    return polyline


def create_ifcextrudedareasolid(ifcfile, point_list, ifcaxis2placement, extrude_dir, extrusion):
    polyline = create_ifcpolyline(ifcfile, point_list)
    ifcclosedprofile = ifcfile.createIfcArbitraryClosedProfileDef("AREA", None, polyline)
    ifcdir = ifcfile.createIfcDirection(extrude_dir)
    ifcextrudedareasolid = ifcfile.createIfcExtrudedAreaSolid(ifcclosedprofile, ifcaxis2placement, ifcdir,
                                                              extrusion) 
    return ifcextrudedareasolid


def colour(ifcfile, entities, color, transparency=0.0):
    """
    统一的材质和颜色/透明度赋予函数
    :param color: [R, G, B] 或者 [R, G, B, Transparency]
    :param transparency: 0.0 为完全不透明，1.0 为完全透明
    """
    if color is not None:
        r, g, b = float(color[0]), float(color[1]), float(color[2])
        
        # 兼容性增强：如果用户在颜色数组里传入了第 4 个值，则自动识别为透明度
        if len(color) >= 4:
            transparency = float(color[3])
            
        # 安全性限制：确保透明度在 0~1 之间
        transparency = max(0.0, min(1.0, float(transparency)))

        owner_history = ifcfile.by_type("IfcOwnerHistory")[0]
        context = ifcfile.by_type("IfcGeometricRepresentationContext")[0]
        
        # 1. 创建基础颜色 RGB
        rgb = ifcfile.create_entity('IfcColourRgb', "SurfaceColor", r, g, b)
        
        # 2. 创建带透明度的高级表面渲染样式 IfcSurfaceStyleRendering
        # IFC 标准参数顺序: SurfaceColour, Transparency, DiffuseColour, TransmissionColour, DiffuseTransmissionColour, ReflectionColour, SpecularColour, SpecularHighlight, ReflectanceMethod
        rendering = ifcfile.create_entity(
            'IfcSurfaceStyleRendering', 
            rgb,                 # SurfaceColour
            transparency,        # Transparency
            None,                # DiffuseColour
            None,                # TransmissionColour
            None,                # DiffuseTransmissionColour
            None,                # ReflectionColour
            None,                # SpecularColour
            None,                # SpecularHighlight
            "NOTDEFINED"         # ReflectanceMethod
        )
        
        # 3. 创建表面样式实体
        style = ifcfile.create_entity('IfcSurfaceStyle', "GeologicalStyle", "BOTH", [rendering])
        
        # 4. 创建材质
        material = ifcfile.create_entity('IfcMaterial', "GeoMaterial")
        
        # 5. 关联材质到实体
        ifcfile.create_entity('IfcRelAssociatesMaterial', ifcopenshell.guid.new(), owner_history, None, None, entities, material)
        
        # 6. 创建样式项并关联到几何上下文
        styled_item = ifcfile.create_entity('IfcStyledItem', None, [style])
        
        # 7. 创建样式表示
        styled_rep = ifcfile.create_entity('IfcStyledRepresentation', context, "Body", "StyledItem", [styled_item])
        
        # 8. 创建材质定义表示
        ifcfile.create_entity('IfcMaterialDefinitionRepresentation', None, None, [styled_rep], material)
    else:
        print('Keep original color')


def generate_ifc_polygon(ifcfile, type, name, _contours, _bottom_height, _wall_height, _scale, color, transparency=0.0):
    _contours = _scale * _contours  
    storey = ifcfile.by_type('IfcBuildingStorey')[0]
    OwnerHistory = ifcfile.by_type('IfcOwnerHistory')[0]
    context = ifcfile.by_type("IfcGeometricRepresentationContext")[0]
    space_placement = create_ifclocalplacement(ifcfile)
    extrusion_placement = create_ifcaxis2placement(ifcfile, point=(0.0, 0.0, float(_bottom_height)))
    
    point_list_extrusion_area = _contours.tolist()
    point_list_extrusion_area.append(_contours[0, :].tolist())  
    
    solid = create_ifcextrudedareasolid(ifcfile, point_list_extrusion_area, extrusion_placement, (0.0, 0.0, 1.0),
                                        float(_wall_height))
    body_representation = ifcfile.createIfcShapeRepresentation(context, "Body", "SweptSolid", [solid]) 
    entity_shape = ifcfile.createIfcProductDefinitionShape(None, None, [body_representation])  
    entity = ifcfile.create_entity(type, ifcopenshell.guid.new())  
    
    entity.OwnerHistory = OwnerHistory
    entity.Name = name
    entity.Representation = entity_shape
    entity.ObjectPlacement = space_placement
    
    related_objects = [entity]
    ifcfile.createIfcRelAggregates(ifcopenshell.guid.new(), OwnerHistory, None, None, storey, related_objects)
    
    # 传递透明度属性
    colour(ifcfile, related_objects, color, transparency)
    return entity


def generate_ifc_borehole(ifcfile, type, name, coordinates, radius, height, direction, color, transparency=0.0):
    storey = ifcfile.by_type('IfcBuildingStorey')[0]
    OwnerHistory = ifcfile.by_type('IfcOwnerHistory')[0]
    context = ifcfile.by_type("IfcGeometricRepresentationContext")[0]
    space_placement = create_ifclocalplacement(ifcfile)
    
    profile = ifcfile.create_entity("IfcCircleProfileDef", ProfileType="AREA", Radius=radius,
                                    Position=ifcfile.create_entity("IfcAxis2Placement2D", Location=ifcfile.create_entity("IfcCartesianPoint", Coordinates=(0.0, 0.0))))
    
    extrusion_placement = ifcfile.create_entity(
        "IfcAxis2Placement3D", Location=ifcfile.create_entity("IfcCartesianPoint", Coordinates=coordinates),
        Axis=ifcfile.create_entity("IfcDirection", DirectionRatios=direction[0]),
        RefDirection=ifcfile.create_entity("IfcDirection", DirectionRatios=direction[1]))
        
    solid = ifcfile.create_entity(
        "IfcExtrudedAreaSolid", SweptArea=profile, Position=extrusion_placement,
        ExtrudedDirection=ifcfile.create_entity("IfcDirection", DirectionRatios=(0.0, 0.0, 1.0)), Depth=height)

    body_representation = ifcfile.createIfcShapeRepresentation(context, "Body", "SweptSolid", [solid])
    entity_shape = ifcfile.createIfcProductDefinitionShape(None, None, [body_representation])
    
    entity = ifcfile.create_entity(type, ifcopenshell.guid.new())
    entity.OwnerHistory = OwnerHistory
    entity.Name = name
    entity.Representation = entity_shape
    entity.ObjectPlacement = space_placement
    
    related_objects = [entity]
    ifcfile.createIfcRelAggregates(ifcopenshell.guid.new(), OwnerHistory, None, None, storey, related_objects)
    
    # 传递透明度属性
    colour(ifcfile, related_objects, color, transparency)
    return entity


def generate_ifc_curve(ifcfile, ifc_type, name, points, color, transparency=0.0):
    """
    将连续的三维点列生成一条 3D IFC 曲线 (1D线条)
    """
    storey = ifcfile.by_type('IfcBuildingStorey')[0]
    OwnerHistory = ifcfile.by_type('IfcOwnerHistory')[0]
    context = ifcfile.by_type("IfcGeometricRepresentationContext")[0]
    space_placement = create_ifclocalplacement(ifcfile)

    # 1. 生成 1D 折线几何对象
    polyline = create_ifcpolyline(ifcfile, points)
    
    # ====== 核心修复：使用 IfcCurveStyle 专门为线条上色 ======
    if color is not None:
        r, g, b = float(color[0]), float(color[1]), float(color[2])
        # 创建专属的 RGB 颜色
        curve_color = ifcfile.create_entity('IfcColourRgb', "LineColor", r, g, b)
        # 创建曲线样式 IfcCurveStyle(Name, CurveFont, CurveWidth, CurveColour)
        # 这里线宽和虚实线型传入 None (由查看软件默认)，强制接管颜色
        curve_style = ifcfile.create_entity('IfcCurveStyle', "AxisCurveStyle", None, None, curve_color)
        # 把样式直接赋给 polyline 这个具体的几何线条，而不是赋给外部构件
        ifcfile.create_entity('IfcStyledItem', polyline, [curve_style])
    # ========================================================
    
    # 2. 封装到形体表达中，标示为 'Curve3D'
    shape_rep = ifcfile.createIfcShapeRepresentation(context, "Axis", "Curve3D", [polyline])
    entity_shape = ifcfile.createIfcProductDefinitionShape(None, None, [shape_rep])

    # 3. 创建 BIM 构件本体
    entity = ifcfile.create_entity(ifc_type, ifcopenshell.guid.new())
    entity.OwnerHistory = OwnerHistory
    entity.Name = name
    entity.Representation = entity_shape
    entity.ObjectPlacement = space_placement

    ifcfile.createIfcRelAggregates(ifcopenshell.guid.new(), OwnerHistory, None, None, storey, [entity])
    
    # 注意：此处不再调用外部的 colour() 函数，因为线条样式已经绑定完毕
    return entity


def add_properties(ifcfile, entity, properties, pset_name):
    existing_pset = ifcopenshell.util.element.get_psets(entity).get(pset_name)
    if existing_pset:
        property_set = ifcfile.by_id(existing_pset["id"])
    else:
        property_set = api.run("pset.add_pset", ifcfile, product=entity, name=pset_name)
    api.run("pset.edit_pset", ifcfile,
            pset=property_set,
            properties=properties
            )


# ==========================================
# 隧道 3D 轴线几何计算与解析函数
# ==========================================
def get_arc_params(x1, y1, x2, y2, bulge):
    dx, dy = x2 - x1, y2 - y1
    L = math.hypot(dx, dy)
    if L < 1e-6 or bulge == 0: return None
    mx, my = x1 + dx / 2.0, y1 + dy / 2.0
    nx, ny = -dy / L, dx / L
    d = (L / 2.0) * (1.0 - bulge**2) / (2.0 * bulge)
    cx, cy = mx + nx * d, my + ny * d
    R = math.hypot(x1 - cx, y1 - cy)
    start_ang = math.atan2(y1 - cy, x1 - cx)
    end_ang = math.atan2(y2 - cy, x2 - cx)
    if bulge > 0:
        sweep_ang = end_ang - start_ang
        while sweep_ang <= 0: sweep_ang += 2 * math.pi
    else:        
        sweep_ang = end_ang - start_ang
        while sweep_ang >= 0: sweep_ang -= 2 * math.pi
    return cx, cy, R, start_ang, sweep_ang

def build_curve_index(points):
    segments = []
    current_dist = 0.0
    for i in range(len(points)-1):
        x1, y1, bulge = points[i]
        x2, y2 = points[i+1][:2]
        if bulge == 0:
            L = math.hypot(x2-x1, y2-y1)
            segments.append({'type': 'line', 'start_dist': current_dist, 'length': L, 'p1': (x1, y1), 'p2': (x2, y2)})
            current_dist += L
        else:
            params = get_arc_params(x1, y1, x2, y2, bulge)
            if params:
                cx, cy, R, start_ang, sweep_ang = params
                L = R * abs(sweep_ang)
                segments.append({'type': 'arc', 'start_dist': current_dist, 'length': L, 'cx': cx, 'cy': cy, 'R': R, 'start_ang': start_ang, 'sweep_ang': sweep_ang})
                current_dist += L
    return segments, current_dist

def get_z_and_slope(profile_data, d):
    if d <= profile_data[0][0]: return profile_data[0][1], 0.0
    if d >= profile_data[-1][0]: return profile_data[-1][1], 0.0
    for i in range(len(profile_data)-1):
        d1, z1 = profile_data[i]
        d2, z2 = profile_data[i+1]
        if d1 <= d <= d2:
            slope = (z2 - z1) / (d2 - d1) if d2 != d1 else 0.0
            return z1 + slope * (d - d1), slope
    return profile_data[-1][1], 0.0

def evaluate_curve_3d(segments, axis_len, profile_data, d):
    pz, slope = get_z_and_slope(profile_data, d)
    d_clipped = max(0.0, min(d, axis_len - 1e-6))
    for seg in segments:
        if seg['start_dist'] <= d_clipped <= seg['start_dist'] + seg['length'] + 1e-6:
            t = (d_clipped - seg['start_dist']) / seg['length']
            if seg['type'] == 'line':
                x1, y1 = seg['p1']; x2, y2 = seg['p2']
                dx, dy = x2 - x1, y2 - y1
                L = math.hypot(dx, dy)
                return x1 + t*dx, y1 + t*dy, pz
            else:
                cx, cy, R = seg['cx'], seg['cy'], seg['R']
                start_ang, sweep_ang = seg['start_ang'], seg['sweep_ang']
                cur_ang = start_ang + t * sweep_ang
                return cx + R*math.cos(cur_ang), cy + R*math.sin(cur_ang), pz
    return None

def generate_ifc_thick_axis(ifcfile, ifc_type, name, points, radius, color, transparency=0.0):
    """
    将 3D 点集生成带有物理宽度的实体圆管（常用于高亮显示轴线）
    :param radius: 圆管的半径（米），控制轴线的“宽度”
    """
    storey = ifcfile.by_type('IfcBuildingStorey')[0]
    OwnerHistory = ifcfile.by_type('IfcOwnerHistory')[0]
    context = ifcfile.by_type("IfcGeometricRepresentationContext")[0]
    space_placement = create_ifclocalplacement(ifcfile)

    solids = []
    # 遍历点集，为每一段线段生成一个圆柱体
    for i in range(len(points)-1):
        p1 = points[i]
        p2 = points[i+1]
        dx, dy, dz = p2[0]-p1[0], p2[1]-p1[1], p2[2]-p1[2]
        length = math.hypot(dx, math.hypot(dy, dz))
        
        if length < 1e-6: continue

        # 计算拉伸方向 (Z轴)
        dir_z = (dx/length, dy/length, dz/length)
        
        # 计算局部 X 轴 (防止奇异矩阵)
        if abs(dir_z[2]) < 0.999:
            ref_x = (-dir_z[1], dir_z[0], 0.0) # 与世界 Z 轴的叉乘
        else:
            ref_x = (1.0, 0.0, 0.0)
            
        norm_x = math.hypot(ref_x[0], ref_x[1])
        ref_x = (ref_x[0]/norm_x, ref_x[1]/norm_x, 0.0)

        # 截面定义：圆形
        profile = ifcfile.create_entity("IfcCircleProfileDef", ProfileType="AREA", Radius=radius,
                                        Position=ifcfile.create_entity("IfcAxis2Placement2D", Location=ifcfile.create_entity("IfcCartesianPoint", Coordinates=(0.0, 0.0))))

        # 局部坐标系
        extrusion_placement = ifcfile.create_entity(
            "IfcAxis2Placement3D", Location=ifcfile.create_entity("IfcCartesianPoint", Coordinates=p1),
            Axis=ifcfile.create_entity("IfcDirection", DirectionRatios=dir_z),
            RefDirection=ifcfile.create_entity("IfcDirection", DirectionRatios=ref_x))

        # 拉伸为实体
        solid = ifcfile.create_entity(
            "IfcExtrudedAreaSolid", SweptArea=profile, Position=extrusion_placement,
            ExtrudedDirection=ifcfile.create_entity("IfcDirection", DirectionRatios=(0.0, 0.0, 1.0)), Depth=length)
        
        solids.append(solid)

    # 将所有圆柱段打包成一个整体几何形体
    body_representation = ifcfile.createIfcShapeRepresentation(context, "Body", "SweptSolid", solids)
    entity_shape = ifcfile.createIfcProductDefinitionShape(None, None, [body_representation])

    entity = ifcfile.create_entity(ifc_type, ifcopenshell.guid.new())
    entity.OwnerHistory = OwnerHistory
    entity.Name = name
    entity.Representation = entity_shape
    entity.ObjectPlacement = space_placement

    related_objects = [entity]
    ifcfile.createIfcRelAggregates(ifcopenshell.guid.new(), OwnerHistory, None, None, storey, related_objects)

    colour(ifcfile, related_objects, color, transparency)
    return entity

def generate_ifc_alignment_4x3(ifcfile, name, axis_pts, profile_data):
    """
    在 IFC4x3 中创建符合基础设施标准的参数化 IfcAlignment 路线骨架
    （修复了 IFC4x3 几何重构及 IfcAlignmentSegment 嵌套体系引起的 IndexError）
    """
    owner_history = ifcfile.by_type("IfcOwnerHistory")[0]
    storey = ifcfile.by_type('IfcBuildingStorey')[0]
    
    # 1. 创建主路线容器
    alignment = ifcfile.create_entity(
        "IfcAlignment",
        GlobalId=ifcopenshell.guid.new(),
        OwnerHistory=owner_history,
        Name=name
    )
    ifcfile.createIfcRelAggregates(ifcopenshell.guid.new(), owner_history, None, None, storey, [alignment])
    
    # 2. 创建平、纵子路线容器
    horizontal_alignment = ifcfile.create_entity(
        "IfcAlignmentHorizontal",
        GlobalId=ifcopenshell.guid.new(),
        OwnerHistory=owner_history,
        Name=f"{name}_Horizontal"
    )
    vertical_alignment = ifcfile.create_entity(
        "IfcAlignmentVertical",
        GlobalId=ifcopenshell.guid.new(),
        OwnerHistory=owner_history,
        Name=f"{name}_Vertical"
    )
    # 将平、纵子容器挂载到主路线上
    ifcfile.createIfcRelAggregates(ifcopenshell.guid.new(), owner_history, None, None, alignment, [horizontal_alignment, vertical_alignment])
    
    # ----------------------------------------------------
    # 3. 写入平面线形段
    # ----------------------------------------------------
    horiz_segments = []
    for i in range(len(axis_pts) - 1):
        x1, y1, bulge = axis_pts[i]
        x2, y2 = axis_pts[i+1][:2]
        dx, dy = x2 - x1, y2 - y1
        L = math.hypot(dx, dy)
        
        # 3.1 提取平面几何核心数学参数
        if bulge == 0:
            seg_type = "LINE"
            start_dir = math.atan2(dy, dx)
            start_radius = 0.0
            end_radius = 0.0
            seg_length = L
        else:
            seg_type = "CIRCULARARC"
            cx, cy, R, start_ang, sweep_ang = get_arc_params(x1, y1, x2, y2, bulge)
            # 圆弧起点的切向角 (取决于凸度正负，即逆时针或顺时针)
            start_dir = start_ang + math.pi/2 if bulge > 0 else start_ang - math.pi/2
            # 依据 IFC 标准：正半径代表逆时针转弯，负半径代表顺时针转弯
            start_radius = R if bulge > 0 else -R
            end_radius = start_radius
            seg_length = R * abs(sweep_ang)
            
        # 3.2 创建【纯参数实体】 (注意：它不再具有 GlobalId 和 Name)
        horiz_param_seg = ifcfile.create_entity(
            "IfcAlignmentHorizontalSegment",
            StartTag=None,
            EndTag=None,
            StartPoint=ifcfile.createIfcCartesianPoint((float(x1), float(y1))),
            StartDirection=float(start_dir),
            StartRadiusOfCurvature=float(start_radius),
            EndRadiusOfCurvature=float(end_radius),
            SegmentLength=float(seg_length),
            GravityCenterLineHeight=None,
            PredefinedType=seg_type
        )
        
        # 3.3 创建包裹该参数的【IFC对象实体】(作为容器，具有 GlobalId)
        horiz_seg_entity = ifcfile.create_entity(
            "IfcAlignmentSegment",
            GlobalId=ifcopenshell.guid.new(),
            OwnerHistory=owner_history,
            Name=f"H_Seg_{i+1}_{seg_type}",
            DesignParameters=horiz_param_seg
        )
        horiz_segments.append(horiz_seg_entity)
        
    # 3.4 将所有的平面分段对象【按前后顺序】嵌套到水平路线中 (使用 IfcRelNests)
    if horiz_segments:
        ifcfile.createIfcRelNests(ifcopenshell.guid.new(), owner_history, None, None, horizontal_alignment, horiz_segments)

    # ----------------------------------------------------
    # 4. 写入纵断面变坡段 
    # ----------------------------------------------------
    vert_segments = []
    for j in range(len(profile_data) - 1):
        d1, z1 = profile_data[j]
        d2, z2 = profile_data[j+1]
        length = d2 - d1
        slope = (z2 - z1) / length if length != 0 else 0.0
        
        # 4.1 创建纵断参数段
        vert_param_seg = ifcfile.create_entity(
            "IfcAlignmentVerticalSegment",
            StartTag=None,
            EndTag=None,
            StartDistAlong=float(d1),
            HorizontalLength=float(length),
            StartHeight=float(z1),
            StartGradient=float(slope),
            EndGradient=float(slope),  # 纯直线变坡段，起点和终点坡度相同
            RadiusOfCurvature=None,    # 直线坡，没有竖曲线曲率
            PredefinedType="CONSTANTGRADIENT"
        )
        
        # 4.2 创建包裹实体
        vert_seg_entity = ifcfile.create_entity(
            "IfcAlignmentSegment",
            GlobalId=ifcopenshell.guid.new(),
            OwnerHistory=owner_history,
            Name=f"V_Seg_{j+1}_CONSTANTGRADIENT",
            DesignParameters=vert_param_seg
        )
        vert_segments.append(vert_seg_entity)
        
    # 4.3 纵断段顺序嵌套
    if vert_segments:
        ifcfile.createIfcRelNests(ifcopenshell.guid.new(), owner_history, None, None, vertical_alignment, vert_segments)
        
    print(f"  Level: IFC4x3 路线母体『{name}』及 {len(axis_pts)-1} 个平面段、{len(profile_data)-1} 个纵断段构建完毕。")
    return alignment


In [3]:
import json
import math
import uuid
import time
import os
import numpy as np
import pandas as pd
import ifcopenshell

# 导入所有通用几何计算与 BIM 生成库


def main():
    # ----------------------------------------------------
    # 1. 隧道数据读取与解析
    # ----------------------------------------------------
    try:
        with open(str(DATA_DIR / "tunnel_axis.json"), 'r') as f: axis_pts = json.load(f)
        with open(str(DATA_DIR / "tunnel_profile.json"), 'r') as f: profile_raw = json.load(f)
    except Exception as e:
        print(f"❌ 读取隧道路线文件失败: {e}")
        return
    
    profile_data = sorted([(item[1], item[0]) for item in profile_raw], key=lambda x: x[0])
    segments, axis_len = build_curve_index(axis_pts)
    
    start_m = max(0.0, profile_data[0][0])
    end_m = min(axis_len, profile_data[-1][0])

    # ----------------------------------------------------
    # 2. 提取 3D 连续轴线 (严格限制在剖面里程内)
    # ----------------------------------------------------
    print("正在解析生成 3D 连续轴线...")
    exact_axis_verts = []
    ARC_RESOLUTION = 1.0 
    
    for seg in segments:
        seg_start = seg['start_dist']
        seg_end = seg_start + seg['length']
        valid_start = max(seg_start, start_m)
        valid_end = min(seg_end, end_m)
        
        if valid_start >= valid_end: continue
            
        if seg['type'] == 'line':
            pt = evaluate_curve_3d(segments, axis_len, profile_data, valid_start)
            if pt: exact_axis_verts.append((pt[0], pt[1], pt[2]))
        elif seg['type'] == 'arc':
            L_valid = valid_end - valid_start
            num_steps = max(2, int(L_valid / ARC_RESOLUTION))
            for i in range(num_steps):
                cur_d = valid_start + (i / num_steps) * L_valid
                pt = evaluate_curve_3d(segments, axis_len, profile_data, cur_d)
                if pt: exact_axis_verts.append((pt[0], pt[1], pt[2]))

    end_pt = evaluate_curve_3d(segments, axis_len, profile_data, end_m)
    if end_pt:
        exact_axis_verts.append((end_pt[0], end_pt[1], end_pt[2]))

    # ----------------------------------------------------
    # 3. 沿轨迹计算 2D 闭环轮廓 (用于生成地质研究域拉伸体)
    # ----------------------------------------------------
    print("正在计算地质研究域的二维基底轮廓...")
    LATERAL_OFFSET = 60.0
    Z_TOP = 0.0
    Z_BOTTOM = -50.0
    
    left_pts, right_pts = [], []
    n_pts = len(exact_axis_verts)
    
    if n_pts >= 2:
        for i in range(n_pts):
            x, y, z = exact_axis_verts[i]
            
            if i == 0:
                dx = exact_axis_verts[1][0] - x
                dy = exact_axis_verts[1][1] - y
            elif i == n_pts - 1:
                dx = x - exact_axis_verts[i-1][0]
                dy = y - exact_axis_verts[i-1][1]
            else:
                dx = exact_axis_verts[i+1][0] - exact_axis_verts[i-1][0]
                dy = exact_axis_verts[i+1][1] - exact_axis_verts[i-1][1]
                
            L2d = math.hypot(dx, dy)
            tx, ty = (1.0, 0.0) if L2d < 1e-6 else (dx/L2d, dy/L2d)
            nx, ny = ty, -tx
            
            x_R = x + LATERAL_OFFSET * nx
            y_R = y + LATERAL_OFFSET * ny
            x_L = x - LATERAL_OFFSET * nx
            y_L = y - LATERAL_OFFSET * ny
            
            left_pts.append([x_L, y_L])
            right_pts.append([x_R, y_R])

    footprint_2d = left_pts + right_pts[::-1]
    contours_array = np.array(footprint_2d)

    # ----------------------------------------------------
    # 4. 初始化 IFC 基础框架与空间结构
    # ----------------------------------------------------
    print("正在初始化 IFC 工程结构...")
    create_guid = lambda: ifcopenshell.guid.compress(uuid.uuid1().hex)
    
    filename = str(OUTPUT_DIR / "tunnel_domain_axis_boreholes_visual.ifc")
    timestamp = round(time.time())
    timestring = time.strftime("%Y-%m-%dT%H:%M:%S", time.gmtime(timestamp))
    
    template = create_template(
        filename=filename, timestamp=timestamp, timestring=timestring, 
        creator="Shu Jiang", organization="WHU", application="IfcOpenShell", 
        application_version="0.7", project_globalid=create_guid(), project_name="Tunnel_Domain"
    )
    

    ifcfile = ifcopenshell.file.from_string(template)  # 在内存中解析 IFC 模板
    owner_history = ifcfile.by_type("IfcOwnerHistory")[0]
    project = ifcfile.by_type("IfcProject")[0]
    
    site_placement = create_ifclocalplacement(ifcfile)
    site = ifcfile.createIfcSite(create_guid(), owner_history, "Site", None, None, site_placement, None, None, "ELEMENT", None, None, None, None, None)
    
    building_placement = create_ifclocalplacement(ifcfile, relative_to=site_placement)
    building = ifcfile.createIfcBuilding(create_guid(), owner_history, 'Building', None, None, building_placement, None, None, "ELEMENT", None, None, None)
    
    storey_placement = create_ifclocalplacement(ifcfile, relative_to=building_placement)
    building_storey = ifcfile.createIfcBuildingStorey(create_guid(), owner_history, 'Storey', None, None, storey_placement, None, None, "ELEMENT", 0.0)
    
    ifcfile.createIfcRelAggregates(create_guid(), owner_history, "Building Container", None, building, [building_storey])
    ifcfile.createIfcRelAggregates(create_guid(), owner_history, "Site Container", None, site, [building])
    ifcfile.createIfcRelAggregates(create_guid(), owner_history, "Project Container", None, project, [site])

    # ----------------------------------------------------
    # 5. 写入 IFC 实体
    # ----------------------------------------------------
    print("正在向 IFC 文件写入地理域及隧道轴线...")
    
    # a. 生成研究域拉伸体
    generate_ifc_polygon(
        ifcfile, 
        type='IfcGeographicElement', 
        name='Study Domain Corridor', 
        _contours=contours_array, 
        _bottom_height=Z_BOTTOM, 
        _wall_height=(Z_TOP - Z_BOTTOM), 
        _scale=1.0, 
        color=[1.0, 0.87, 0.68, 0.7]
    )
    
    # 在 tunnel2bim.py 中修改：
    generate_ifc_thick_axis(
        ifcfile, 
        ifc_type='IfcBuildingElementProxy', 
        name='Continuous Axis Curve', 
        points=exact_axis_verts, 
        radius=0.2,            # <--- 设为 0.2 米（20厘米），看起来就像一条线
        color=[1.0, 0.0, 0.0],  # 纯红色，Blender 绝对能完美显示！
        transparency=0
    )

    # ----------------------------------------------------
    # 6. 读取表格，生成地质钻孔 (原 Notebook 逻辑重现)
    # ----------------------------------------------------
    excel_path = str(PROJECT_DIR / "Datasource_dikan_new.xlsx")
    if os.path.exists(excel_path):
        print(f"\n找到地勘数据 {excel_path}，正在生成钻孔及属性...")
        
        # 读取钻孔基础信息
        borehole_coordinates = pd.read_excel(excel_path, sheet_name='Borehole_coordinates').values.astype(float)
        borehole_depth = pd.read_excel(excel_path, sheet_name='Borehole_depth').values.astype(float)
        
        # 读取地质层物理力学属性
        df_prop = pd.read_excel(excel_path, sheet_name="Property", index_col=0)
        geo_data = tuple({prop: value for prop, value in df_prop[col].dropna().items()} for col in df_prop.columns)
        
        radius = 1.0
        direction = [(0.0, 0.0, 1.0), (1.0, 0.0, 0.0)]
        layer_colors = ([1, 0.973, 0.863], [0, 0.749, 1], [0.545, 0.451, 0.333], [1, 0.843, 0], [0, 0.604, 0.804], [0.063, 0.306, 0.545])
        
        for i, row in enumerate(borehole_coordinates):
            depth = borehole_depth[i].astype(float)
            for j in range(len(depth)-1):
                # 钻孔坐标Z值为当前分层底标高 (注意原逻辑为 -depth[j+1])
                coordinates = (float(row[0]), float(row[1]), float(-depth[j+1]))
                height = depth[j+1] - depth[j]
                
                # 容错：防止层数超过预设颜色列表
                current_color = layer_colors[j] if j < len(layer_colors) else [0.7, 0.7, 0.7]
                
                # 生成圆柱体实体 (可按需将 IfcBuildingElementProxy 换为原先的 IfcAirTerminalBox)
                entity = generate_ifc_borehole(
                    ifcfile, 'IfcBuildingElementProxy', f'Borehole_{i+1}_Layer_{j+1}', 
                    coordinates, radius, height, direction, current_color
                )
                
                # 为对应地层关联属性集 (Pset)
                if j < len(geo_data):
                    add_properties(ifcfile, entity, geo_data[j], "CustomPropertySet")
                    
            print(f" ✅ 完成第 {i+1} 个钻孔创建")
    else:
        print(f"\n⚠️ 警告: 当前目录下未找到 {excel_path} 文件，将跳过生成钻孔。")

    # ----------------------------------------------------
    # 7. 最终导出
    # ----------------------------------------------------
    ifcfile.write(filename)
    print(f"\n🎉 自动化建模完毕！已生成标准 BIM 模型文件：{filename}")


if __name__ == "__main__":
    main()

正在解析生成 3D 连续轴线...
正在计算地质研究域的二维基底轮廓...
正在初始化 IFC 工程结构...
正在向 IFC 文件写入地理域及隧道轴线...

找到地勘数据 E:\Github\geo_inves_bim_model\Datasource_dikan_new.xlsx，正在生成钻孔及属性...
 ✅ 完成第 1 个钻孔创建
 ✅ 完成第 2 个钻孔创建
 ✅ 完成第 3 个钻孔创建
 ✅ 完成第 4 个钻孔创建
 ✅ 完成第 5 个钻孔创建
 ✅ 完成第 6 个钻孔创建
 ✅ 完成第 7 个钻孔创建
 ✅ 完成第 8 个钻孔创建
 ✅ 完成第 9 个钻孔创建
 ✅ 完成第 10 个钻孔创建
 ✅ 完成第 11 个钻孔创建
 ✅ 完成第 12 个钻孔创建
 ✅ 完成第 13 个钻孔创建
 ✅ 完成第 14 个钻孔创建
 ✅ 完成第 15 个钻孔创建
 ✅ 完成第 16 个钻孔创建
 ✅ 完成第 17 个钻孔创建
 ✅ 完成第 18 个钻孔创建
 ✅ 完成第 19 个钻孔创建
 ✅ 完成第 20 个钻孔创建
 ✅ 完成第 21 个钻孔创建
 ✅ 完成第 22 个钻孔创建
 ✅ 完成第 23 个钻孔创建
 ✅ 完成第 24 个钻孔创建
 ✅ 完成第 25 个钻孔创建
 ✅ 完成第 26 个钻孔创建
 ✅ 完成第 27 个钻孔创建
 ✅ 完成第 28 个钻孔创建
 ✅ 完成第 29 个钻孔创建
 ✅ 完成第 30 个钻孔创建
 ✅ 完成第 31 个钻孔创建
 ✅ 完成第 32 个钻孔创建
 ✅ 完成第 33 个钻孔创建
 ✅ 完成第 34 个钻孔创建
 ✅ 完成第 35 个钻孔创建
 ✅ 完成第 36 个钻孔创建
 ✅ 完成第 37 个钻孔创建
 ✅ 完成第 38 个钻孔创建
 ✅ 完成第 39 个钻孔创建
 ✅ 完成第 40 个钻孔创建
 ✅ 完成第 41 个钻孔创建
 ✅ 完成第 42 个钻孔创建
 ✅ 完成第 43 个钻孔创建
 ✅ 完成第 44 个钻孔创建
 ✅ 完成第 45 个钻孔创建
 ✅ 完成第 46 个钻孔创建

🎉 自动化建模完毕！已生成标准 BIM 模型文件：E:\Github\geo_inves_bim_model\tunnel_domain_axis_boreholes_visual.ifc


In [4]:
import json
import math
import uuid
import time
import os
import numpy as np
import pandas as pd
import ifcopenshell

# 导入所有通用几何计算与 BIM 生成库


def main():
    # ----------------------------------------------------
    # 1. 隧道数据读取与解析
    # ----------------------------------------------------
    try:
        with open(str(DATA_DIR / "tunnel_axis.json"), 'r') as f: axis_pts = json.load(f)
        with open(str(DATA_DIR / "tunnel_profile.json"), 'r') as f: profile_raw = json.load(f)
    except Exception as e:
        print(f"❌ 读取隧道路线文件失败: {e}")
        return
    
    profile_data = sorted([(item[1], item[0]) for item in profile_raw], key=lambda x: x[0])
    segments, axis_len = build_curve_index(axis_pts)
    
    start_m = max(0.0, profile_data[0][0])
    end_m = min(axis_len, profile_data[-1][0])

    # ----------------------------------------------------
    # 2. 提取 3D 连续轴线 (严格限制在剖面里程内)
    # ----------------------------------------------------
    print("正在解析生成 3D 连续轴线...")
    exact_axis_verts = []
    ARC_RESOLUTION = 1.0 
    
    for seg in segments:
        seg_start = seg['start_dist']
        seg_end = seg_start + seg['length']
        valid_start = max(seg_start, start_m)
        valid_end = min(seg_end, end_m)
        
        if valid_start >= valid_end: continue
            
        if seg['type'] == 'line':
            pt = evaluate_curve_3d(segments, axis_len, profile_data, valid_start)
            if pt: exact_axis_verts.append((pt[0], pt[1], pt[2]))
        elif seg['type'] == 'arc':
            L_valid = valid_end - valid_start
            num_steps = max(2, int(L_valid / ARC_RESOLUTION))
            for i in range(num_steps):
                cur_d = valid_start + (i / num_steps) * L_valid
                pt = evaluate_curve_3d(segments, axis_len, profile_data, cur_d)
                if pt: exact_axis_verts.append((pt[0], pt[1], pt[2]))

    end_pt = evaluate_curve_3d(segments, axis_len, profile_data, end_m)
    if end_pt:
        exact_axis_verts.append((end_pt[0], end_pt[1], end_pt[2]))

    # ----------------------------------------------------
    # 3. 沿轨迹计算 2D 闭环轮廓 (用于生成地质研究域拉伸体)
    # ----------------------------------------------------
    print("正在计算地质研究域的二维基底轮廓...")
    LATERAL_OFFSET = 60.0
    Z_TOP = 0.0
    Z_BOTTOM = -50.0
    
    left_pts, right_pts = [], []
    n_pts = len(exact_axis_verts)
    
    if n_pts >= 2:
        for i in range(n_pts):
            x, y, z = exact_axis_verts[i]
            
            if i == 0:
                dx = exact_axis_verts[1][0] - x
                dy = exact_axis_verts[1][1] - y
            elif i == n_pts - 1:
                dx = x - exact_axis_verts[i-1][0]
                dy = y - exact_axis_verts[i-1][1]
            else:
                dx = exact_axis_verts[i+1][0] - exact_axis_verts[i-1][0]
                dy = exact_axis_verts[i+1][1] - exact_axis_verts[i-1][1]
                
            L2d = math.hypot(dx, dy)
            tx, ty = (1.0, 0.0) if L2d < 1e-6 else (dx/L2d, dy/L2d)
            nx, ny = ty, -tx
            
            x_R = x + LATERAL_OFFSET * nx
            y_R = y + LATERAL_OFFSET * ny
            x_L = x - LATERAL_OFFSET * nx
            y_L = y - LATERAL_OFFSET * ny
            
            left_pts.append([x_L, y_L])
            right_pts.append([x_R, y_R])

    footprint_2d = left_pts + right_pts[::-1]
    contours_array = np.array(footprint_2d)

    # ----------------------------------------------------
    # 4. 初始化 IFC 基础框架与空间结构
    # ----------------------------------------------------
    print("正在初始化 IFC 工程结构...")
    create_guid = lambda: ifcopenshell.guid.compress(uuid.uuid1().hex)
    
    filename = str(OUTPUT_DIR / "tunnel_domain_axis_boreholes.ifc")
    timestamp = round(time.time())
    timestring = time.strftime("%Y-%m-%dT%H:%M:%S", time.gmtime(timestamp))
    
    template = create_template(
        filename=filename, timestamp=timestamp, timestring=timestring, 
        creator="Shu Jiang", organization="WHU", application="IfcOpenShell", 
        application_version="0.7", project_globalid=create_guid(), project_name="Tunnel_Domain"
    )
    

    ifcfile = ifcopenshell.file.from_string(template)  # 在内存中解析 IFC 模板
    owner_history = ifcfile.by_type("IfcOwnerHistory")[0]
    project = ifcfile.by_type("IfcProject")[0]
    
    site_placement = create_ifclocalplacement(ifcfile)
    site = ifcfile.createIfcSite(create_guid(), owner_history, "Site", None, None, site_placement, None, None, "ELEMENT", None, None, None, None, None)
    
    building_placement = create_ifclocalplacement(ifcfile, relative_to=site_placement)
    building = ifcfile.createIfcBuilding(create_guid(), owner_history, 'Building', None, None, building_placement, None, None, "ELEMENT", None, None, None)
    
    storey_placement = create_ifclocalplacement(ifcfile, relative_to=building_placement)
    building_storey = ifcfile.createIfcBuildingStorey(create_guid(), owner_history, 'Storey', None, None, storey_placement, None, None, "ELEMENT", 0.0)
    
    ifcfile.createIfcRelAggregates(create_guid(), owner_history, "Building Container", None, building, [building_storey])
    ifcfile.createIfcRelAggregates(create_guid(), owner_history, "Site Container", None, site, [building])
    ifcfile.createIfcRelAggregates(create_guid(), owner_history, "Project Container", None, project, [site])

    # ----------------------------------------------------
    # 5. 写入 IFC 实体
    # ----------------------------------------------------
    print("正在向 IFC 文件写入地理域及隧道轴线...")
    
    # a. 生成研究域拉伸体 (使用你原本正确的完整参数)
    generate_ifc_polygon(
        ifcfile, 
        type='IfcGeographicElement', 
        name='Study Domain Corridor', 
        _contours=contours_array, 
        _bottom_height=Z_BOTTOM, 
        _wall_height=(Z_TOP - Z_BOTTOM), 
        _scale=1.0, 
        color=[1.0, 0.87, 0.68, 0.2]
    )
    
    # b. 生成用于三维渲染与可视化的 3D 折线（在 OpenBIM 软件中直接可见）
    generate_ifc_curve(
        ifcfile, 
        ifc_type='IfcBuildingElementProxy', 
        name='Continuous Axis Curve (Visual)', 
        points=exact_axis_verts, 
        color=[1.0, 0.0, 0.0],  # 纯红色
        transparency=0.0
    )
    
    # c. 生成符合 IFC4x3 标准的真实参数化路线（用于后续隧道结构高级建模）
    generate_ifc_alignment_4x3(
        ifcfile,
        name='Main_Tunnel_Alignment_4x3',
        axis_pts=axis_pts,
        profile_data=profile_data
    )

    # ----------------------------------------------------
    # 6. 读取表格，生成地质钻孔 (原 Notebook 逻辑重现)
    # ----------------------------------------------------
    excel_path = str(PROJECT_DIR / "Datasource_dikan_new.xlsx")
    if os.path.exists(excel_path):
        print(f"\n找到地勘数据 {excel_path}，正在生成钻孔及属性...")
        
        # 读取钻孔基础信息
        borehole_coordinates = pd.read_excel(excel_path, sheet_name='Borehole_coordinates').values.astype(float)
        borehole_depth = pd.read_excel(excel_path, sheet_name='Borehole_depth').values.astype(float)
        
        # 读取地质层物理力学属性
        df_prop = pd.read_excel(excel_path, sheet_name="Property", index_col=0)
        geo_data = tuple({prop: value for prop, value in df_prop[col].dropna().items()} for col in df_prop.columns)
        
        radius = 1.0
        direction = [(0.0, 0.0, 1.0), (1.0, 0.0, 0.0)]
        layer_colors = ([1, 0.973, 0.863], [0, 0.749, 1], [0.545, 0.451, 0.333], [1, 0.843, 0], [0, 0.604, 0.804], [0.063, 0.306, 0.545])
        
        for i, row in enumerate(borehole_coordinates):
            depth = borehole_depth[i].astype(float)
            for j in range(len(depth)-1):
                # 钻孔坐标Z值为当前分层底标高 (注意原逻辑为 -depth[j+1])
                coordinates = (float(row[0]), float(row[1]), float(-depth[j+1]))
                height = depth[j+1] - depth[j]
                
                # 容错：防止层数超过预设颜色列表
                current_color = layer_colors[j] if j < len(layer_colors) else [0.7, 0.7, 0.7]
                
                # 生成圆柱体实体 (可按需将 IfcBuildingElementProxy 换为原先的 IfcAirTerminalBox)
                entity = generate_ifc_borehole(
                    ifcfile, 'IfcBuildingElementProxy', f'Borehole_{i+1}_Layer_{j+1}', 
                    coordinates, radius, height, direction, current_color
                )
                
                # 为对应地层关联属性集 (Pset)
                if j < len(geo_data):
                    add_properties(ifcfile, entity, geo_data[j], "CustomPropertySet")
                    
            print(f" ✅ 完成第 {i+1} 个钻孔创建")
    else:
        print(f"\n⚠️ 警告: 当前目录下未找到 {excel_path} 文件，将跳过生成钻孔。")

    # ----------------------------------------------------
    # 7. 最终导出
    # ----------------------------------------------------
    ifcfile.write(filename)
    print(f"\n🎉 自动化建模完毕！已生成标准 BIM 模型文件：{filename}")


if __name__ == "__main__":
    main()

正在解析生成 3D 连续轴线...
正在计算地质研究域的二维基底轮廓...
正在初始化 IFC 工程结构...
正在向 IFC 文件写入地理域及隧道轴线...
  Level: IFC4x3 路线母体『Main_Tunnel_Alignment_4x3』及 634 个平面段、4 个纵断段构建完毕。

找到地勘数据 E:\Github\geo_inves_bim_model\Datasource_dikan_new.xlsx，正在生成钻孔及属性...
 ✅ 完成第 1 个钻孔创建
 ✅ 完成第 2 个钻孔创建
 ✅ 完成第 3 个钻孔创建
 ✅ 完成第 4 个钻孔创建
 ✅ 完成第 5 个钻孔创建
 ✅ 完成第 6 个钻孔创建
 ✅ 完成第 7 个钻孔创建
 ✅ 完成第 8 个钻孔创建
 ✅ 完成第 9 个钻孔创建
 ✅ 完成第 10 个钻孔创建
 ✅ 完成第 11 个钻孔创建
 ✅ 完成第 12 个钻孔创建
 ✅ 完成第 13 个钻孔创建
 ✅ 完成第 14 个钻孔创建
 ✅ 完成第 15 个钻孔创建
 ✅ 完成第 16 个钻孔创建
 ✅ 完成第 17 个钻孔创建
 ✅ 完成第 18 个钻孔创建
 ✅ 完成第 19 个钻孔创建
 ✅ 完成第 20 个钻孔创建
 ✅ 完成第 21 个钻孔创建
 ✅ 完成第 22 个钻孔创建
 ✅ 完成第 23 个钻孔创建
 ✅ 完成第 24 个钻孔创建
 ✅ 完成第 25 个钻孔创建
 ✅ 完成第 26 个钻孔创建
 ✅ 完成第 27 个钻孔创建
 ✅ 完成第 28 个钻孔创建
 ✅ 完成第 29 个钻孔创建
 ✅ 完成第 30 个钻孔创建
 ✅ 完成第 31 个钻孔创建
 ✅ 完成第 32 个钻孔创建
 ✅ 完成第 33 个钻孔创建
 ✅ 完成第 34 个钻孔创建
 ✅ 完成第 35 个钻孔创建
 ✅ 完成第 36 个钻孔创建
 ✅ 完成第 37 个钻孔创建
 ✅ 完成第 38 个钻孔创建
 ✅ 完成第 39 个钻孔创建
 ✅ 完成第 40 个钻孔创建
 ✅ 完成第 41 个钻孔创建
 ✅ 完成第 42 个钻孔创建
 ✅ 完成第 43 个钻孔创建
 ✅ 完成第 44 个钻孔创建
 ✅ 完成第 45 个钻孔创建
 ✅ 完成第 46 个钻孔创建

🎉 自动化建模完毕！已生成标准 BIM 模型文件：E:\Github\geo_inves_

In [5]:
import ifcopenshell
import math

def extract_alignment_info(ifc_file_path):
    print(f"正在读取文件: {ifc_file_path}")
    ifcfile = ifcopenshell.open(ifc_file_path)

    # 1. 查找 IFC 模型中所有的 IfcAlignment 实体
    alignments = ifcfile.by_type("IfcAlignment")
    if not alignments:
        print("未在模型中找到 IfcAlignment 实体！")
        return

    for alignment in alignments:
        print(f"\n{'='*50}")
        print(f"🎯 找到路线: {alignment.Name} (GlobalId: {alignment.GlobalId})")
        print(f"{'='*50}")

        # 2. 遍历这个路线所“聚合(RelAggregates)”的所有子对象 (平纵线形容器)
        for rel_agg in alignment.IsDecomposedBy:
            for sub_alignment in rel_agg.RelatedObjects:
                
                # ------ A. 提取平面线形 ------
                if sub_alignment.is_a("IfcAlignmentHorizontal"):
                    print(f"\n📍 【平面线形】容器名称: {sub_alignment.Name}")
                    
                    # 平面段是通过 "RelNests" 嵌套的
                    if hasattr(sub_alignment, "IsNestedBy") and sub_alignment.IsNestedBy:
                        for rel_nest in sub_alignment.IsNestedBy:
                            for seg_entity in rel_nest.RelatedObjects:
                                if seg_entity.is_a("IfcAlignmentSegment"):
                                    # 提取真正的数学参数实体
                                    param = seg_entity.DesignParameters
                                    if param and param.is_a("IfcAlignmentHorizontalSegment"):
                                        print(f"  - 分段名称: {seg_entity.Name}")
                                        print(f"    类型: {param.PredefinedType}")
                                        
                                        # 提取起点坐标
                                        if param.StartPoint:
                                            coords = param.StartPoint.Coordinates
                                            print(f"    起点坐标: (X: {coords[0]:.2f}, Y: {coords[1]:.2f})")
                                            
                                        # 提取切向角 (弧度转度)
                                        if param.StartDirection is not None:
                                            dir_deg = math.degrees(param.StartDirection)
                                            print(f"    起点切向角: {dir_deg:.2f}°")
                                            
                                        # 提取长度和半径
                                        if param.SegmentLength is not None:
                                            print(f"    段长: {param.SegmentLength:.2f} m")
                                        if param.StartRadiusOfCurvature is not None and param.StartRadiusOfCurvature != 0:
                                            print(f"    曲率半径: {param.StartRadiusOfCurvature:.2f} m")
                                        print(f"    {'-'*20}")

                # ------ B. 提取纵断面线形 ------
                elif sub_alignment.is_a("IfcAlignmentVertical"):
                    print(f"\n🏔️ 【纵断面线形】容器名称: {sub_alignment.Name}")
                    
                    if hasattr(sub_alignment, "IsNestedBy") and sub_alignment.IsNestedBy:
                        for rel_nest in sub_alignment.IsNestedBy:
                            for seg_entity in rel_nest.RelatedObjects:
                                if seg_entity.is_a("IfcAlignmentSegment"):
                                    param = seg_entity.DesignParameters
                                    if param and param.is_a("IfcAlignmentVerticalSegment"):
                                        print(f"  - 分段名称: {seg_entity.Name}")
                                        print(f"    类型: {param.PredefinedType}")
                                        if param.StartDistAlong is not None:
                                            print(f"    起始里程: K{param.StartDistAlong/1000:.3f} ({param.StartDistAlong:.2f} m)")
                                        if param.StartHeight is not None:
                                            print(f"    起始高程: {param.StartHeight:.2f} m")
                                        if param.HorizontalLength is not None:
                                            print(f"    水平段长: {param.HorizontalLength:.2f} m")
                                        if param.StartGradient is not None:
                                            print(f"    坡度: {param.StartGradient*100:.2f} %")
                                        print(f"    {'-'*20}")

# 执行提取测试
# 请将这里的路径换成你刚刚生成的那个 IFC 文件的绝对路径
ifc_path = str(OUTPUT_DIR / "tunnel_domain_axis_boreholes.ifc")
extract_alignment_info(ifc_path)

正在读取文件: E:\Github\geo_inves_bim_model\tunnel_domain_axis_boreholes.ifc

🎯 找到路线: Main_Tunnel_Alignment_4x3 (GlobalId: 0QRFgfQTD5luT42qTjQkHX)

📍 【平面线形】容器名称: Main_Tunnel_Alignment_4x3_Horizontal
  - 分段名称: H_Seg_1_LINE
    类型: LINE
    起点坐标: (X: 0.00, Y: 0.00)
    起点切向角: 75.64°
    段长: 1.00 m
    --------------------
  - 分段名称: H_Seg_2_LINE
    类型: LINE
    起点坐标: (X: 0.25, Y: 0.97)
    起点切向角: 75.64°
    段长: 1.00 m
    --------------------
  - 分段名称: H_Seg_3_LINE
    类型: LINE
    起点坐标: (X: 0.50, Y: 1.94)
    起点切向角: 75.64°
    段长: 1.00 m
    --------------------
  - 分段名称: H_Seg_4_LINE
    类型: LINE
    起点坐标: (X: 0.74, Y: 2.91)
    起点切向角: 75.65°
    段长: 1.00 m
    --------------------
  - 分段名称: H_Seg_5_LINE
    类型: LINE
    起点坐标: (X: 0.99, Y: 3.88)
    起点切向角: 75.65°
    段长: 1.00 m
    --------------------
  - 分段名称: H_Seg_6_LINE
    类型: LINE
    起点坐标: (X: 1.24, Y: 4.84)
    起点切向角: 75.66°
    段长: 1.00 m
    --------------------
  - 分段名称: H_Seg_7_LINE
    类型: LINE
    起点坐标: (X: 1.49, Y: 5.81)
    